In [108]:
# the parameters from MCMC results to form the parameters file for simulations

import pandas as pd

def read_nml_file(file_path):
    """ read nml file content """
    with open(file_path, "r") as file:
        lines = file.readlines()
    return lines

def parse_nml_lines(lines):
    """ parse the lines in a nml-format with multiple groups """
    data = {}
    current_group = None
    for line in lines:
        line = line.strip()
        if line.startswith('&'): # start a new group
            current_group = line[1:]
            data[current_group] = {}
        elif line.startswith('/'): # end current group
            current_group = None
        elif "=" in line and current_group:
            key, value = line.split('=', 1)
            data[current_group][key.strip()] = value.strip()
    return data

def update_nml_data(data, group, updates):
    """ update the nml date for the specified group, where the updates is a dictionary containing the key-value pairs to be updated  """
    if group in data:
        for key, value in updates.items():
            if key in data[group]:
                data[group][key] = value

def write_nml_file(file_path, data):
    """ re-write the updated nml data back to the file, keeping the group structure """
    with open(file_path, 'w') as file:
        for group, items in data.items():
            file.write(f'&{group}\n')
            for key, value in items.items():
                file.write(f'{key} = {value}\n')
            file.write('/\n')


def run_params4nml(file_params, org_data, file_path):
    # read the parameter file 
    df_dat = pd.read_csv(file_params).iloc[-1]
    print("test len: ", len(df_dat))
    print(df_dat)

    df_dict = {}
    for idx in df_dat.index:
        if idx.split("_")[-1].rstrip() in ["Tree", "Shrub", "Sphagnum"]:
            item,_,_ = idx.rpartition('_')
            if item == "SLA": item = "SLAx"
            if item in df_dict.keys():
                df_dict[item] = df_dict[item] + ", " + str(df_dat[idx])
            else:
                df_dict[item] = str(df_dat[idx])
        else:
            df_dict[idx] = str(df_dat[idx])
    print(df_dict)
    # update
    i = 0
    nml_name = ["nml_site_params", "nml_species_params"]
    # group_to_update = 'nml_species_params'  # 
    for group_to_update in nml_name:
        for ikey, item in df_dict.items():
            updates = {ikey.strip(): item}  # the content for updating
            update_nml_data(org_data, group_to_update, updates)
            # print(updates)
    # print(i)

    write_nml_file(file_path, org_data)



ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]

org_file = "parameters.nml" # destination file for nml-format
# ls_plots = ["P04", "P06", "P08", "P10", "P13", "P16", "P17", "P19", "P20"]
ls_plots = ["P06"]

# each plot
for idx_plot, iplot in enumerate(ls_plots):
    print(iplot)
    file_params = f"../../outputs/run_mcmc_pretreat_{iplot}/results_mcmc/total_parameter_sets_001.csv"
    file_path   = f"../inputs/in_pretreat/{iplot}/parameters.nml"#f"inputs/in_pretreat/{iplot}/parameters.nml"
    lines = read_nml_file(file_path)
    data  = parse_nml_lines(lines)
    run_params4nml(file_params, data, file_path)

P06
test len:  302
 Tau_F               0.7027
Tau_C               16.3000
Tau_Micro            0.7140
Tau_SlowSOM         43.7000
Tau_Passive        410.7000
                     ...   
fn2r_4_Sphagnum      0.3364
fn2r_5_Sphagnum      0.3364
fn2r_6_Sphagnum      0.3364
fn2r_7_Sphagnum      0.3364
fn2r_8_Sphagnum      0.3364
Name: 9, Length: 302, dtype: float64
{' Tau_F': '0.7027', 'Tau_C': '16.3', 'Tau_Micro': '0.714', 'Tau_SlowSOM': '43.7', 'Tau_Passive': '410.7', 'Q10rh': '5.114', 'f_F2M': '0.3851', 'f_C2M': '0.2584', 'f_C2S': '0.2794', 'f_M2S': '0.3131', 'f_M2P': '0.009478', 'f_S2P': '0.03111', 'f_S2M': '0.4092', 'f_P2M': '0.3004', 'r_me': '0.09669', 'Q10pro': '4.124', 'Omax': '5.157', 'CH4_thre': '940.5', 'Tveg': '7.34', 'f': '0.1189', 'bubprob': '0.4541', 'Vmaxfraction': '0.02625', 'f_slow': '0.5004', 's_soil': '1.009', 'pox': '0.4039', 'f_lit_m': '0.261', 's_lit': '0.322', 'Q10rh_1': '2.49', 'Q10rh_2': '3.66', 'Q10rh_3': '5.433', 'Q10rh_4': '4.188', 'Q10rh_5': '6.394', 'Q10rh_6'

In [109]:
import os
import f90nml

# update the parameter values in ["nml_site_params", "nml_species_params"] to mcmc_conf

def handle_data(iplot):
    # Define file paths
    parameters_nml_path = f"../inputs/in_pretreat/{iplot}/parameters.nml" #f"inputs/in_treat/{iplot}/parameters.nml"
    mcmc_conf_nml_path  = f"../configs/conf_pretreat/mcmc_conf_{iplot}.nml" #"mcmc_conf_P06.nml" #"configs/mcmc_conf.nml"

    # Read parameters.nml content
    nml_params_data    = f90nml.read(parameters_nml_path)
    nml_mcmc_conf_data = f90nml.read(mcmc_conf_nml_path)

    # dict paramter
    dict_site_params = dict(nml_params_data["nml_site_params"])
    dict_spp_params  = dict(nml_params_data["nml_species_params"])

    # modify mcmc_conf data
    for idx_nml, inml in enumerate(nml_mcmc_conf_data["siteDAParams"]["st"]):
        if inml["parname"].lower() in dict_site_params.keys():
            nml_mcmc_conf_data["siteDAParams"]["st"][idx_nml]["parVal"] = dict_site_params[inml["parname"].lower()]

    for idx_sp, isp in enumerate(nml_mcmc_conf_data["spDAParams"]["sp"]):
        for idx_var, ivar in enumerate(isp["var"]):
            if ivar["parname"].lower() in dict_spp_params.keys():
                # if idx_sp < len(nml_mcmc_conf_data["spDAParams"]["sp"]):
                #     print(f"len(nml_mcmc_conf_data['spDAParams']['sp'][idx_sp]['var']): {len(nml_mcmc_conf_data['spDAParams']['sp'][idx_sp]['var'])}")
                print(ivar["parname"].lower(),dict_spp_params[ivar["parname"].lower()])
                nml_mcmc_conf_data["spDAParams"]["sp"][idx_sp]["var"][idx_var]["parval"] = dict_spp_params[ivar["parname"].lower()][idx_sp]

    # write updated mcmc_conf to file
    nml_mcmc_conf_data.write(mcmc_conf_nml_path, force=True)#, mode="w")

ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]
ls_plots = ["P06"]
for idx, iplot in enumerate(ls_plots):
    print(iplot)
    handle_data(iplot)

P06
laimax [4.08, 5.636, 7.737]
laimin [0.3, 0.3, 0.3]
stom_n [2, 2, 2]
saps [0.4495, 0.4969, 0.4884]
sapr [1.0, 1.0, 1.0]
slax [165.8, 16.55, 182.7]
glmax [12.98, 16.63, 28.92]
grmax [14.44, 21.1, 13.33]
gsmax [7.034, 11.6, 32.58]
alpha [0.385, 0.385, 0.385]
vcmax0 [14.74, 24.99, 87.62]
ds0 [2000, 2000, 2000]
xfang [0, 0, 0]
rdepth [150, 150, 150]
rootmax [500, 500, 500]
stemmax [1000, 1000, 1000]
tau_leaf [2.143, 2.013, 0.04332]
tau_stem [60.13, 58.55, 0.5255]
tau_root [0.3656, 2.443, 0.2188]
q10 [1.392, 1.776, 1.255]
rl0 [41.58, 61.57, 43.08]
rs0 [6.153, 12.05, 54.2]
rr0 [28.89, 12.3, 26.81]
jv [1.08, 1.342, 2.347]
entrpy [674.7, 617.8, 677.0]
gddonset [124.7, 145.7, 93.72]
hmax [24.19, 24.19, 24.19]
hl0 [0.00019, 0.00019, 0.00019]
laimax0 [8.0, 8.0, 8.0]
la0 [0.2, 0.2, 0.2]
fn2l [0.5408, 0.3378, 0.3439]
fn2r [0.2384, 0.0571, 0.3349]
s_cleaf [1.358, 0.5477, 1.846]
s_cstem [0.2039, 0.1001, 3.493]
s_croot [2.592, 3.373, 9.673]
s_nsc [1.901, 3.158, 9.734]
s_nsn [0.6504, 5.447, 7.527]
f